# 1. One dimensional dynamics
This Notebook aims to take the code from the previous Notebook and turn it one dimensional.

## 1.1) Defining the lattice
We impose closed boundary conditions (meaning the edges count as empty cells) and each cell in the lattice constitutes the lattice lenght and has a probability of being activated. Both the lattice length and probability of activation are input variables.

Additionally, the remaining cells are subjected to a probability that they will be inert cells or volatile. The higher probability, the higher chance that the cell will be volatile and thus, not inert.

## 1.2) Defining the activation mechanisms
Just like previously, there are two activation mechanisms: Short range and long range.

The short range activation mechanism can only activate one neighboring cell at a time. The probability is a discrete value q and the activation process involves a probability of q/2 between choosing right or left and then an arbitrary porobability of the activation process actually going through.

The long range activation mechanism works such that it most likely activates a neighboring cell and the propbabiloity dies down as the distance increases, and can only lead to a maximum single activation per activated cell.

With this information in place we can recompute the dynamics and history functions:

In [ ]:
def distance_cells_1d(y, k, L):
        cells = []
        for ny in (y - k, y + k):
            if 0 <= ny < L:
                cells.append(ny)
        return cells

In [ ]:
import numpy as np

def dynamics_1d(rho, f_k, p_succ_long, p_succ_short=0.9, p_long=0.5,
                       m=1, L=200, steps=200, rng=None):
    
    rng = rng or np.random.default_rng()

    grid = (rng.random(L) < rho).astype(int)
    state = np.where(grid == 1, 0, -1)

    xs = np.where(grid == 1)[0]
    if len(xs) == 0:
        return state
    seed = rng.integers(len(xs))
    state[xs[seed]] = 1

    for t in range(steps):
        fired = np.argwhere(state == 1).flatten()
        if len(fired) == 0:
            break

        new_fires = []
        for fx in fired:
            for _ in range(m):
                if rng.random() < p_long:
                    k = f_k(rng)
                    if k < 2:
                        continue
                    p_hit = p_succ_long(k)
                else:
                    k = 1
                    p_hit = p_succ_short

                candidates = distance_cells_1d(fx, k, L)
                if not candidates:
                    continue
                tx = candidates[rng.integers(len(candidates))]
                if state[tx] == 0 and rng.random() < p_hit:
                    new_fires.append(tx)

        state[state == 1] = -2

        if not new_fires:
            break
        for tx in set(new_fires):
            state[tx] = 1

    return state

In [ ]:
def history_1d(rho, f_k, p_succ_long, p_succ_short=0.9,
                      p_long=0.5, m=1, L=200, steps=200, rng=None):
    rng = rng or np.random.default_rng()
    grid = (rng.random(L) < rho).astype(int)
    state = np.where(grid == 1, 0, -1)
    xs = np.where(grid == 1)[0]
    if len(xs) == 0:
        return [state.copy()]
    seed = rng.integers(len(xs))
    state[xs[seed]] = 1

    snapshots = [state.copy()]          # save initial frame

    for t in range(steps):
        fired = np.argwhere(state == 1).flatten()
        if len(fired) == 0:
            break
        new_fires = []
        for fx in fired:
            for _ in range(m):
                if rng.random() < p_long:
                    k = f_k(rng)
                    if k < 2:
                        continue
                    p_hit = p_succ_long(k)
                else:
                    k = 1
                    p_hit = p_succ_short
                candidates = distance_cells_1d(fx, k, L)
                if not candidates:
                    continue
                tx = candidates[rng.integers(len(candidates))]
                if state[tx] == 0 and rng.random() < p_hit:
                    new_fires.append(tx)
        state[state == 1] = -2
        if not new_fires:
            snapshots.append(state.copy())
            break
        for tx in set(new_fires):
            state[tx] = 1
        snapshots.append(state.copy())   # save frame after this timestep

    return snapshots

## 1.3) Plotting the rest of the functions
The rest of the functions are easier to adapt, because they are not affected by the shape of the grid:

In [ ]:
# show_timestep, plot_alive, plot_deactivated_vs_density_time, plot_all_settings_vs_density_time

